In [ ]:
# ---------------- Imports ----------------
import json
import os
from collections import defaultdict

import pandas as pd
import yaml
from sklearn.metrics import confusion_matrix, recall_score
import numpy as np


import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib as mpl

# Path to font
FONT_DIR = os.path.join("../../config", "fonts", "linux_libertine")
FONT_PATH = os.path.join(FONT_DIR, "LinLibertine_R.ttf")

# Register font
fm.fontManager.addfont(FONT_PATH)

# Get font name
libertine_font = fm.FontProperties(fname=FONT_PATH).get_name()

# Set globally
mpl.rcParams.update({
    "font.family": libertine_font,
    "pdf.fonttype": 42,
})
    


In [ ]:
# ---------------- Args ----------------


#### Llama 3.1 8B @ 0.3 NO FEEDBACK

RUNTYPE = "no-feedback"
MODEL_CHOICE = "meta-llama/Llama-3.1-8B-Instruct"
output_file_name = "llama-3.1-8b-instruct-combined-claims-15k-0.3-100x100trajs-no-feedback"
RESULTS_FILES = {
    "baseline": [
        "20260427t012417-20260128T2129-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-1-100x100trajs-nofeedback",
        "20260427t014136-20260130T1707-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-1-100x100trajs-nofeedback",
        "20260427t015855-20260130T1730-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-1-100x100trajs-nofeedback",

    ],
    "authoritative": [
        "20260427t021615-20260130T1258-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-0.3-100x100trajs-nofeedback",
        "20260427t023336-20260130T1617-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-0.3-100x100trajs-nofeedback",
        "20260427t025054-20260130T1641-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-authoritative-0.3-100x100trajs-nofeedback",

    ],
    "consensus": [
        "20260427t030810-20260131T1055-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-consensus-0.3-100x100trajs-nofeedback",
        "20260427t032529-20260131T1118-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-consensus-0.3-100x100trajs-nofeedback",
        "20260427t034252-20260131T1141-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-consensus-0.3-100x100trajs-nofeedback",

    ],
    #"emotional": [
    #    "20260427t040013-20260131T1314-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-emotional-0.3-100x100trajs-nofeedback",
    #    "20260427t041731-20260131T1336-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-emotional-0.3-100x100trajs-nofeedback",
    #    "20260427t043449-20260131T1359-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-emotional-0.3-100x100trajs-nofeedback",
    #    
    #],
    "prestige": [
        "20260427t045206-20260131T1206-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-prestige-0.3-100x100trajs-nofeedback",
        "20260427t050920-20260131T1229-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-prestige-0.3-100x100trajs-nofeedback",
        "20260427t052643-20260131T1251-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-prestige-0.3-100x100trajs-nofeedback",
    
    ],
    "sensationalist": [
        "20260427t054402-20260131T1421-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-sensationalist-0.3-100x100trajs-nofeedback",
        "20260427t060120-20260131T1444-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-sensationalist-0.3-100x100trajs-nofeedback",
        "20260427t061837-20260131T1506-llama-3.1-8b-instruct-20260115T095923-combined-claims-15k-sensationalist-0.3-100x100trajs-nofeedback",

    ],
}









In [ ]:
# ---------------- Config ----------------

with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

PROJ_STORE = config["paths"]["proj-store"]

if RUNTYPE == "no-feedback":
    RESULTS_FOLDER = os.path.join(PROJ_STORE, "experiments", "model-evaluate-trajectory", "no-feedback", MODEL_CHOICE)
elif RUNTYPE == "feedback":
    RESULTS_FOLDER = os.path.join(PROJ_STORE, "experiments", "model-evaluate-trajectory", "feedback", MODEL_CHOICE)
else:
    raise ValueError(f"Invalid RUNTYPE: {RUNTYPE}")


# OUTPUT


if RUNTYPE == "no-feedback":
    OUTPUT_DIR = os.path.join(PROJ_STORE, "evaluation", "trajectories-results", "no-feedback", MODEL_CHOICE)
elif RUNTYPE == "feedback":
    OUTPUT_DIR = os.path.join(PROJ_STORE, "evaluation", "trajectories-results", "feedback", MODEL_CHOICE)
else:
    raise ValueError(f"Invalid RUNTYPE: {RUNTYPE}")




os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_FILE = os.path.join(OUTPUT_DIR, f"{output_file_name}")
                          
                          
SCORE_TYPES = ["linear", "log"]


In [ ]:
# -------------------------
# Functions
# -------------------------

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


def make_output_base(score_type):
    return os.path.join(OUTPUT_DIR, f"{output_file_name}-{score_type}")

In [ ]:
# ---------------- Load all results once ----------------

all_rows = []

for model_target, fnames in RESULTS_FILES.items():
    for fname in fnames:
        path = os.path.join(RESULTS_FOLDER, f"{fname}.jsonl")

        for row in load_jsonl(path):
            row["model_framing_target"] = model_target
            row["run_id"] = fname
            all_rows.append(row)

all_df = pd.DataFrame(all_rows)

display(all_df.head())
display(all_df.shape)

In [ ]:

# ---------------- Shared confidence summaries ----------------

df_all = all_df.copy()

df_all["is_correct"] = df_all["predicted_label"] == df_all["true_label"]
df_all["confidence"] = df_all.apply(
    lambda r: r["scores"][r["predicted_label"]]["word_cond_prob"],
    axis=1
)

df_all["conf_correct"] = np.where(df_all["is_correct"], df_all["confidence"], np.nan)
df_all["conf_incorrect"] = np.where(~df_all["is_correct"], df_all["confidence"], np.nan)




# ---------------- Error counts per framing ----------------

error_df = df_all.copy()

# mark errors
error_df["is_error"] = error_df["predicted_label"] != error_df["true_label"]

# count errors per run and framing
errors_per_run = (
    error_df
    .groupby(["model_framing_target", "run_id", "framing_type"])["is_error"]
    .sum()
    .reset_index(name="n_errors")
)


# pivot → one column per framing
errors_pivot = (
    errors_per_run
    .pivot_table(
        index=["model_framing_target", "run_id"],
        columns="framing_type",
        values="n_errors",
        fill_value=0
    )
    .reset_index()
)

# average across runs
errors_summary = (
    errors_pivot
    .groupby("model_framing_target")
    .mean(numeric_only=True)
    .reset_index()
)
errors_summary = errors_summary.rename(
    columns={
        col: f"errors_{col}"
        for col in errors_summary.columns
        if col != "model_framing_target"
    }
)

# ---------------- Encounters ----------------
encounters_per_run = (
    error_df
    .groupby(["model_framing_target", "run_id", "framing_type"])
    .size()
    .reset_index(name="n_encounters")
)

encounters_pivot = (
    encounters_per_run
    .pivot_table(
        index=["model_framing_target", "run_id"],
        columns="framing_type",
        values="n_encounters",
        fill_value=0
    )
    .reset_index()
)

encounters_summary = (
    encounters_pivot
    .groupby("model_framing_target")
    .mean(numeric_only=True)
    .reset_index()
)

encounters_summary = encounters_summary.rename(
    columns={
        col: f"encounters_{col}"
        for col in encounters_summary.columns
        if col != "model_framing_target"
    }
)




# ---------------- Merge + Error Rates ----------------
errors_summary = errors_summary.merge(
    encounters_summary,
    on="model_framing_target",
    how="left"
)

for col in errors_summary.columns:
    if col.startswith("errors_"):
        framing = col.replace("errors_", "")
        enc_col = f"encounters_{framing}"
        rate_col = f"error_rate_{framing}"

        if enc_col in errors_summary.columns:
            errors_summary[rate_col] = (
                errors_summary[col] / errors_summary[enc_col]
            )




ORDER_WITH_BASELINE_FIRST = ["baseline"] + sorted(
    [k for k in errors_summary["model_framing_target"].unique() if k != "baseline"]
)

errors_summary["model_framing_target"] = pd.Categorical(
    errors_summary["model_framing_target"],
    categories=ORDER_WITH_BASELINE_FIRST,
    ordered=True
)

errors_summary = errors_summary.sort_values("model_framing_target")


display(errors_summary)


per_run = (
    df_all
    .groupby(["model_framing_target", "run_id"])
    .agg(
        n_supports_encountered=("true_label", lambda x: (x == "SUPPORTS").sum()),
        n_refutes_encountered=("true_label", lambda x: (x == "REFUTES").sum()),
        P_correct=("is_correct", "mean"),
        mean_conf_correct=("conf_correct", "mean"),
        mean_conf_incorrect=("conf_incorrect", "mean"),
    )
    .reset_index()
)

summary = (
    per_run
    .groupby("model_framing_target", as_index=False)
    .mean(numeric_only=True)
)

summary["P_incorrect"] = 1.0 - summary["P_correct"]
summary["confidence_gap"] = summary["mean_conf_correct"] - summary["mean_conf_incorrect"]


summary["model_framing_target"] = pd.Categorical(
    summary["model_framing_target"],
    categories=ORDER_WITH_BASELINE_FIRST,
    ordered=True
)

summary = summary.sort_values("model_framing_target")

display(summary)
summary.to_csv(os.path.join(OUTPUT_DIR, f"{output_file_name}-summary.csv"), index=False)



In [ ]:
# ---------------- CS processing for both metrics ----------------

ORDER = ["baseline", "authoritative", "consensus", "emotional", "prestige", "sensationalist"]
LABEL_MAP = {k: k.capitalize() for k in ORDER}

cmap = plt.get_cmap("tab10")
colors = cmap.colors


cs_results = {}

for SCORE_TYPE in SCORE_TYPES:
    RT_COL = f"r_t_{SCORE_TYPE}"
    HT_COL = f"H_t_{SCORE_TYPE}"
    OUTPUT_FILE = make_output_base(SCORE_TYPE)

    print(f"Processing {SCORE_TYPE}")

    # ---------------- Benchmark ----------------
    orig_df = all_df[all_df["model_framing_target"] == "baseline"].copy()

    orig_traj = (
        orig_df
        .sort_values(["run_id", "trajectory_id", "step"])
        .groupby(["run_id", "trajectory_id"])
        .agg(
            T=("step", "count"),
            final_H=(HT_COL, "last"),
            mean_H=(HT_COL, "mean"),
        )
        .reset_index()
    )

    orig_traj["mean_H_per_step"] = orig_traj["final_H"] / orig_traj["T"]

    benchmark = {
        "p25": np.percentile(orig_traj["mean_H_per_step"], 25),
        "p50": np.percentile(orig_traj["mean_H_per_step"], 50),
        "p75": np.percentile(orig_traj["mean_H_per_step"], 75),
        "mean": orig_traj["mean_H_per_step"].mean(),
        "std": orig_traj["mean_H_per_step"].std(),
    }

    display(pd.DataFrame([benchmark]))
    pd.DataFrame([benchmark]).to_csv(f"{OUTPUT_FILE}-benchmark.csv", index=False)

    # ---------------- Mean CS per step plot ----------------
    step_summary = (
        all_df
        .groupby(["model_framing_target", "run_id", "step"])[HT_COL]
        .mean()
        .reset_index()
    )

    step_summary = (
        step_summary
        .groupby(["model_framing_target", "step"])[HT_COL]
        .mean()
        .reset_index()
    )

    step_summary["t"] = step_summary["step"] + 1
    step_summary["H_per_step"] = step_summary[HT_COL] / step_summary["t"]

    present_keys = [
        k for k in ORDER
        if k in step_summary["model_framing_target"].unique()
    ]

    zero_rows = []

    for key in present_keys:
        zero_rows.append({
            "model_framing_target": key,
            "step": -1,
            HT_COL: 0.0,
            "t": 0,
            "H_per_step": 0.0
        })

    zero_df = pd.DataFrame(zero_rows)

    step_summary = pd.concat([step_summary, zero_df], ignore_index=True)
    step_summary = step_summary.sort_values(["model_framing_target", "step"])


    plt.figure(figsize=(7, 4))

    for i, key in enumerate(present_keys):
        g = step_summary[step_summary["model_framing_target"] == key].sort_values("step")
        linestyle = "--" if key == "baseline" else "-"

        plt.plot(
            g["t"],
            g["H_per_step"],
            label=LABEL_MAP[key],
            color=colors[i],
            linestyle=linestyle,
            linewidth=2
        )

    plt.xlabel("Step $t$", fontsize=22)
    plt.ylabel(f"Mean CS ({SCORE_TYPE})", fontsize=22)
    plt.legend(loc="upper right", fontsize=16)
    plt.xticks(fontsize=22)
    plt.yticks(fontsize=22)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_FILE}-cs-average.pdf", format="pdf", bbox_inches="tight")
    plt.show()

    # ---------------- Cumulative CS plot ----------------
    cum_summary = (
        all_df
        .groupby(["model_framing_target", "run_id", "step"])[HT_COL]
        .mean()
        .reset_index()
    )

    cum_summary = (
        cum_summary
        .groupby(["model_framing_target", "step"])[HT_COL]
        .mean()
        .reset_index()
    )

    plt.figure(figsize=(7, 4))

    for i, key in enumerate(present_keys):
        g = cum_summary[cum_summary["model_framing_target"] == key].sort_values("step")
        linestyle = "--" if key == "baseline" else "-"

        plt.plot(
            g["step"],
            g[HT_COL],
            label=LABEL_MAP[key],
            color=colors[i],
            linestyle=linestyle,
            linewidth=2
        )

    plt.xlabel("Step $t$")
    plt.ylabel(f"CS ({SCORE_TYPE})")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_FILE}-cs-cumulative.pdf", format="pdf", bbox_inches="tight")
    plt.show()

    # ---------------- Trajectory summaries ----------------
    traj_summary = (
        all_df
        .sort_values(["model_framing_target", "run_id", "trajectory_id", "step"])
        .groupby(["model_framing_target", "run_id", "trajectory_id"])
        .agg(
            T=("step", "count"),
            final_H=(HT_COL, "last"),
            mean_H=(HT_COL, "mean"),
        )
        .reset_index()
    )

    # Per-step CS 
    traj_summary["CS_per_step"] = traj_summary["final_H"] / traj_summary["T"]

    # Total CS 
    traj_summary["CS_total"] = traj_summary["final_H"]

    display(traj_summary.head())
    traj_summary.to_csv(f"{OUTPUT_FILE}-traj-summary.csv", index=False)

    mean_CS = (
        traj_summary
        .groupby(["model_framing_target", "run_id"])
        .agg(
            CS_per_step=("CS_per_step", "mean"),
            CS_total=("CS_total", "mean"),
        )
        .reset_index()
    )

    mean_CS = (
        mean_CS
        .groupby("model_framing_target")
        .mean(numeric_only=True)
        .reset_index()
    )

    # Rename per score type
    mean_CS = mean_CS.rename(columns={
        "CS_per_step": f"CS_{SCORE_TYPE}_per_step",
        "CS_total": f"CS_{SCORE_TYPE}_total",
    })

    cs_results[SCORE_TYPE] = mean_CS

    display(mean_CS)
    mean_CS.to_csv(f"{OUTPUT_FILE}-mean-cs.csv", index=False)
    
    
    
    
    # ---------------- Mean cumulative CS (average H_t over time) ----------------
    # This computes:
    #   mean_cumulative_CS = (1 / T) * sum_t H_t
    # where H_t is already cumulative.
    #
    # Note:
    # - This is NOT the final CS H_T.
    # - This is the average value of the cumulative score over time.

    mean_cum_traj_summary = (
        all_df
        .sort_values(["model_framing_target", "run_id", "trajectory_id", "step"])
        .groupby(["model_framing_target", "run_id", "trajectory_id"])
        .agg(
            T=("step", "max"),
            sum_H=(HT_COL, "sum"),
        )
        .reset_index()
    )

    mean_cum_traj_summary["mean_cumulative_CS"] = (
        mean_cum_traj_summary["sum_H"] / mean_cum_traj_summary["T"]
    )

    display(mean_cum_traj_summary.head())
    mean_cum_traj_summary.to_csv(
        f"{OUTPUT_FILE}-mean-cumulative-traj-summary.csv",
        index=False
    )

    mean_cum_CS = (
        mean_cum_traj_summary
        .groupby(["model_framing_target", "run_id"])["mean_cumulative_CS"]
        .mean()
        .reset_index()
    )

    mean_cum_CS = (
        mean_cum_CS
        .groupby("model_framing_target")["mean_cumulative_CS"]
        .mean()
        .reset_index(name=f"mean_cumulative_CS_{SCORE_TYPE}")
    )

    display(mean_cum_CS)
    mean_cum_CS.to_csv(
        f"{OUTPUT_FILE}-mean-cumulative-cs.csv",
        index=False
    )

# Merge CS tables
cs_linear = cs_results["linear"]
cs_log = cs_results["log"]

final_table = summary.merge(cs_linear, on="model_framing_target", how="left")
final_table = final_table.merge(cs_log, on="model_framing_target", how="left")

final_table = final_table[
    [
        "model_framing_target",
        "n_supports_encountered",
        "n_refutes_encountered",
        "P_correct",
        "mean_conf_correct",
        "mean_conf_incorrect",
        "P_incorrect",
        "CS_linear_total",
        "CS_log_total",
        "CS_linear_per_step",
        "CS_log_per_step",
    ]
]


# enforce ordering
final_table["model_framing_target"] = pd.Categorical(
    final_table["model_framing_target"],
    categories=ORDER_WITH_BASELINE_FIRST,
    ordered=True
)


final_table = final_table.merge(
    errors_summary,
    on="model_framing_target",
    how="left"
)


final_table = final_table.sort_values("model_framing_target")

display(final_table)

final_table.to_csv(
    os.path.join(OUTPUT_DIR, f"{output_file_name}-full-summary.csv"),
    index=False
)




In [ ]:
fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(7, 5),
    sharex=True
)

for ax, SCORE_TYPE in zip(axes, ["linear", "log"]):

    HT_COL = f"H_t_{SCORE_TYPE}"

    step_summary = (
        all_df
        .groupby(["model_framing_target", "run_id", "step"])[HT_COL]
        .mean()
        .reset_index()
    )

    step_summary = (
        step_summary
        .groupby(["model_framing_target", "step"])[HT_COL]
        .mean()
        .reset_index()
    )

    # EXACT same logic as original working plots
    step_summary["t"] = step_summary["step"] + 1
    step_summary["H_per_step"] = step_summary[HT_COL] / step_summary["t"]

    present_keys = [
        k for k in ORDER
        if k in step_summary["model_framing_target"].unique()
    ]

    # ---- Synthetic zero row (CRITICAL) ----
    zero_rows = []
    for key in present_keys:
        zero_rows.append({
            "model_framing_target": key,
            "step": -1,
            HT_COL: 0.0,
            "t": 0,
            "H_per_step": 0.0
        })

    zero_df = pd.DataFrame(zero_rows)

    step_summary = pd.concat([step_summary, zero_df], ignore_index=True)
    step_summary = step_summary.sort_values(["model_framing_target", "step"])

    # ---- Plot ----
    for i, key in enumerate(present_keys):
        g = step_summary[
            step_summary["model_framing_target"] == key
        ].sort_values("step")

        linestyle = "--" if key == "baseline" else "-"

        ax.plot(
            g["t"],
            g["H_per_step"],
            label=LABEL_MAP[key],
            color=colors[i],
            linestyle=linestyle,
            linewidth=2
        )

    ax.set_ylabel(f"Mean CS ({SCORE_TYPE})", fontsize=20)
    ax.tick_params(axis="both", labelsize=20)

# ---- Labels + legend ----
axes[1].set_xlabel("Step $t$", fontsize=20)

axes[0].legend(
    loc="upper right",
    fontsize=16,
    labelspacing=0,
    borderpad=0,
)

plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, f"{output_file_name}-combined-cs-average.pdf"),
    format="pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# %%
# -------------------------------------------------------
# Statistical summary table for final cumulative scores
# -------------------------------------------------------

BOOTSTRAP_SAMPLES = 5000

stat_rows = []

for SCORE_TYPE in ["linear", "log"]:

    HT_COL = f"H_t_{SCORE_TYPE}"

    traj_final = (
        all_df
        .sort_values(
            ["model_framing_target", "run_id", "trajectory_id", "step"]
        )
        .groupby(
            ["model_framing_target", "run_id", "trajectory_id"]
        )
        .agg(
            final_H=(HT_COL, "last")
        )
        .reset_index()
    )

    for key in ORDER:

        if key not in traj_final["model_framing_target"].unique():
            continue

        vals = traj_final[
            traj_final["model_framing_target"] == key
        ]["final_H"].values

        mean_val = np.mean(vals)
        std_val = np.std(vals)

        # ---------------------------------------------------
        # Bootstrap CI
        # ---------------------------------------------------

        boot_means = []

        for _ in range(BOOTSTRAP_SAMPLES):

            sample = np.random.choice(
                vals,
                size=len(vals),
                replace=True
            )

            boot_means.append(np.mean(sample))

        ci_low = np.percentile(boot_means, 2.5)
        ci_high = np.percentile(boot_means, 97.5)

        stat_rows.append({
            "score_type": SCORE_TYPE,
            "model_framing_target": key,
            "mean_final_CS": mean_val,
            "std_final_CS": std_val,
            "ci95_low": ci_low,
            "ci95_high": ci_high,
            "n_trajectories": len(vals),
        })

# -------------------------------------------------------
# Create dataframe
# -------------------------------------------------------

stats_df = pd.DataFrame(stat_rows)

stats_df["model_framing_target"] = pd.Categorical(
    stats_df["model_framing_target"],
    categories=ORDER,
    ordered=True
)

stats_df = stats_df.sort_values(
    ["score_type", "model_framing_target"]
)

display(stats_df)

# -------------------------------------------------------
# Save CSV
# -------------------------------------------------------

stats_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        f"{output_file_name}-final-score-statistics.csv"
    ),
    index=False
)
